# Prepare SCI data
In this notebook, we select the necessary emission-related variables from the Scenario Compass Database

In [5]:
import pyam
import pandas as pd

import dotenv
import os
from pathlib import Path

import datetime

In [6]:
dotenv.load_dotenv()

# Set up output folder
OUTPUT_FOLDER = Path(os.environ["OUTPUT_FOLDER"])
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

In [8]:
df = pyam.IamDataFrame(
    Path(
        os.environ['SCI_DATA']
    )
)

[INFO] 19:09:43 - pyam.core: Reading file /Users/hoegner/Projects/data/SCI/SCI-2025_v1.0_pathways_ensemble_global.xlsx


We have a lot of variables available but only need relevant emission-related variables for this analysis.
For this, we downselect the variables that are present in the historical emissions data that is used for CMIP7: https://zenodo.org/records/17845154

In [20]:
hist_data = pyam.IamDataFrame(
    Path(
        os.environ['HIST_DATA']
    )
)

[INFO] 23:26:53 - pyam.core: Reading file /Users/hoegner/Projects/data/historical/cmip7-hist/processed/global-workflow-history_202511261223_202511040855_202512032146_202512021030_7e32405ade790677a6022ff498395bff00d9792d_202511040855_202512071232_202511040855_202511040855_0002_0002.csv


We check for the intersection of the two sets of variables.

In [10]:
variable_intersection = list(
    set(df.variable).intersection(hist_data.variable)
)

What is present?

In [11]:
variable_intersection, len(variable_intersection), len(hist_data.variable)

(['Emissions|BC',
  'Emissions|CO2|Energy and Industrial Processes',
  'Emissions|HFC|HFC227ea',
  'Emissions|N2O',
  'Emissions|HFC|HFC23',
  'Emissions|HFC|HFC134a',
  'Emissions|C2F6',
  'Emissions|CO2|AFOLU',
  'Emissions|C6F14',
  'Emissions|NOx',
  'Emissions|HFC|HFC245fa',
  'Emissions|Sulfur',
  'Emissions|HFC|HFC32',
  'Emissions|HFC|HFC43-10',
  'Emissions|CH4',
  'Emissions|CF4',
  'Emissions|HFC|HFC143a',
  'Emissions|HFC|HFC125',
  'Emissions|VOC',
  'Emissions|OC',
  'Emissions|SF6',
  'Emissions|NH3',
  'Emissions|CO'],
 23,
 52)

What is missing?

In [12]:
[x for x in hist_data.variable if x not in variable_intersection]

['Emissions|C3F8',
 'Emissions|C4F10',
 'Emissions|C5F12',
 'Emissions|C7F16',
 'Emissions|C8F18',
 'Emissions|CCl4',
 'Emissions|CFC11',
 'Emissions|CFC113',
 'Emissions|CFC114',
 'Emissions|CFC115',
 'Emissions|CFC12',
 'Emissions|CH2Cl2',
 'Emissions|CH3Br',
 'Emissions|CH3CCl3',
 'Emissions|CH3Cl',
 'Emissions|CHCl3',
 'Emissions|HCFC141b',
 'Emissions|HCFC142b',
 'Emissions|HCFC22',
 'Emissions|HFC|HFC152a',
 'Emissions|HFC|HFC236fa',
 'Emissions|HFC|HFC365mfc',
 'Emissions|Halon1202',
 'Emissions|Halon1211',
 'Emissions|Halon1301',
 'Emissions|Halon2402',
 'Emissions|NF3',
 'Emissions|SO2F2',
 'Emissions|cC4F8']

Seems fine so far. These will be handled later..

In [13]:
df_downselected = (
    df
    .filter(
        variable=variable_intersection
    )
)

In [14]:
df_downselected.require_data(
    year=2100,
    exclude_on_fail=True
)

[INFO] 19:10:30 - pyam.validation: 22 scenarios failed validation and will be set as `exclude=True`.


,model,scenario,year
0,IMACLIM 2.0,NAVIGATE Structural Change-Baseline,2100
1,IMACLIM 2.0,NAVIGATE Structural Change-Net-Zero,2100
2,PROMETHEUS 1.2,ECEMF-DIAG-C400-lin,2100
3,PROMETHEUS 1.2,ECEMF-DIAG-NPi,2100
4,PROMETHEUS V1,IAM-COMPACT-Current-Policies-Emissions-Intensity,2100
5,PROMETHEUS V1,IAM-COMPACT-NDC-Emissions-Intensity,2100
6,PROMETHEUS V1,IAM-COMPACT-NDC-Long-Term-Pledges,2100
7,PROMETHEUS V1,NAVIGATE Demand-1.5°C-act_u,2100
8,PROMETHEUS V1,NAVIGATE Demand-1.5°C-all_u,2100
9,PROMETHEUS V1,NAVIGATE Demand-1.5°C-ele_u,2100


In [15]:
df_downselected.filter(
    model='IMACLIM 2.0', #Check later why this is happening...
    keep=False,
    inplace=True
)

Let us interpolate for missing years and save this out for further analysis

In [16]:
df_downselected.interpolate(
    time=range(2010, 2101),
    inplace=True
)

Ok, for some reason, it appears that the HFC32 go *negative* for some scenarios.
Get rid of those...

In [17]:
df_downselected.validate(
    variable='Emissions|HFC|HFC32',
    lower_bound=0,
    exclude_on_fail=True
)

[WARNING] 19:10:55 - pyam.validation: 728 of 2101975 data points do not satisfy the criteria
[INFO] 19:10:55 - pyam.validation: 27 scenarios failed validation and will be set as `exclude=True`.


,model,scenario,region,variable,unit,year,value
0,MESSAGE-GLOBIOM 1.0,ADVANCE-2020-1.5°C-2100,World,Emissions|HFC|HFC32,kt HFC32/yr,2070,-1.300000e-04
1,MESSAGE-GLOBIOM 1.0,ADVANCE-2020-1.5°C-2100,World,Emissions|HFC|HFC32,kt HFC32/yr,2071,-1.345800e-04
2,MESSAGE-GLOBIOM 1.0,ADVANCE-2020-1.5°C-2100,World,Emissions|HFC|HFC32,kt HFC32/yr,2072,-1.391600e-04
3,MESSAGE-GLOBIOM 1.0,ADVANCE-2020-1.5°C-2100,World,Emissions|HFC|HFC32,kt HFC32/yr,2073,-1.437400e-04
4,MESSAGE-GLOBIOM 1.0,ADVANCE-2020-1.5°C-2100,World,Emissions|HFC|HFC32,kt HFC32/yr,2074,-1.483200e-04
...,...,...,...,...,...,...,...
723,MESSAGEix-GLOBIOM GEI 1.0,GEI-SSP2-openres-lc-CB400,World,Emissions|HFC|HFC32,kt HFC32/yr,2070,-4.448093e-15
724,MESSAGEix-GLOBIOM GEI 1.0,GEI-SSP2-openres-lc-CB450,World,Emissions|HFC|HFC32,kt HFC32/yr,2070,-4.448093e-15
725,MESSAGEix-GLOBIOM GEI 1.0,GEI-SSP2-openres-lc-CB500,World,Emissions|HFC|HFC32,kt HFC32/yr,2070,-4.448093e-15
726,MESSAGEix-GLOBIOM GEI 1.0,GEI-SSP2-openres-lc-CB550,World,Emissions|HFC|HFC32,kt HFC32/yr,2070,-4.448093e-15


In [18]:
df_downselected.filter(
    exclude=False,
    inplace=True
)

In [19]:
df_downselected.to_csv(
    OUTPUT_FOLDER / f'100_inputdata_{datetime.date.today().strftime("%Y%m%d")}.csv'
)